Initialization and loading input

In [1]:
import os
from cimgraph.databases import XMLFile
from cimgraph.models import FeederModel
import cimgraph.data_profile.cimhub_2023 as cim
os.environ['CIMG_CIM_PROFILE'] = 'cimhub_2023'
file = XMLFile(filename='../test_models/IEEE13.xml')
feeder = cim.Feeder(mRID= '49AD8E07-3BF9-A4E2-CB8F-C3722F837B62')
container = feeder
network = FeederModel(container=feeder, connection=file)

In [2]:
import logging
import json 
from cimgraph.models import GraphModel
from __future__ import annotations


from cimgraph.databases import get_cim_profile
_log = logging.getLogger(__name__)

def catalog_parser(catalog_file, network):
    file = open(catalog_file) ## opening json file
    catalog = json.load(file) ## loading jsonj file contents as catalog
    data = catalog['catalog'] ## catalog dictionary, first element (in this example, only single XF) saved into data
    cim_profile, cim = get_cim_profile() # Import CIM profile 
    obj = item_parser(data, network, cim) ## validating and simplifying json file objects into a single structure of obj. - needs modifications
    file.close()
    return obj

def item_parser(data:dict, network: GraphModel, cim):
    class_type = edge_class = eval(f'cim.{data["@type"]}')
    obj = class_type()
    network.add_to_graph(obj)

    for attribute in data:
        if type(data[attribute]) == str:
            if attribute in class_type.__dataclass_fields__:
                setattr(obj, attribute, data[attribute])
            else:
                _log.warning(f'Attribute {attribute} not found')
        elif type(data[attribute]) == list:
            if attribute in class_type.__dataclass_fields__:
                values = getattr(obj, attribute)
                for item in data[attribute]:
                    value = item_parser(item, network, cim)
                    values.append(value)
                setattr(obj, attribute, values)
    return obj

Catalog_JSON_file_path = '../test_models/hv69_12.json' ### Power Transformer json file path
Obj_CP_output = catalog_parser(Catalog_JSON_file_path, network) ### Catalog file reads the input and returns an object 

Attribute @type not found
Attribute @type not found
Attribute x not found
Attribute b not found
Attribute @type not found
Attribute x not found
Attribute b not found


In [3]:
catalog_file = Catalog_JSON_file_path
file = open(catalog_file) ## opening json file
catalog = json.load(file) ## loading jsonj file contents as catalog
data = catalog['catalog'] ## catalog dictionary, first element (in this example, only single XF) saved into data
cim_profile, cim = get_cim_profile() # Import CIM profile 

print(data)
obj = item_parser(data, network, cim) ## validating and simplifying json file objects into a single structure of obj. - needs modifications
file.close()

Attribute @type not found
Attribute @type not found
Attribute x not found
Attribute b not found
Attribute @type not found
Attribute x not found
Attribute b not found


{'@type': 'PowerTransformer', 'name': 'hvmv69_12', 'vectorGroup': 'Yy', 'PowerTransformerEnd': [{'@type': 'PowerTransformerEnd', 'name': 'hvmv69_12_End_1', 'endNumber': '1', 'grounded': 'true', 'rground': '0', 'xground': '0', 'connectionKind': 'WindingConnection.Y', 'phaseAngleClock': '0', 'r': '1.594935', 'x': '18.90117', 'b': '0', 'ratedS': '20000000', 'ratedU': '69000'}, {'@type': 'PowerTransformerEnd', 'name': 'hvmv69_12_End_2', 'endNumber': '2', 'grounded': 'true', 'rground': '0', 'xground': '0', 'connectionKind': 'WindingConnection.Y', 'phaseAngleClock': '0', 'r': '0.052092802', 'x': '0', 'b': '0', 'ratedS': '20000000', 'ratedU': '12470'}]}
